# Laboratório — Problemas de Satisfação de Restrições (CSP)

O código está completo. Execute, altere os parâmetros assinalados e responda às **cinco questões**. Não precisa de escrever funções.

## Como trabalhar
1. Abra este notebook no Google Colab e guarde uma cópia. Não é necessária GPU nem instalar bibliotecas.
2. Execute a preparação e, depois, as experiências pela ordem apresentada.
3. Altere apenas os parâmetros marcados com **ALTERAR** e volte a executar a célula.
4. Preencha as células **Resposta** com resultados observados e justificações.
5. Entregue o notebook com as respostas e os resultados guardados. Consulte o histórico no final.

**Duração prevista:** 60–90 minutos.

## Conceitos e regras comuns
- Um **domínio** contém as opções ainda disponíveis; uma opção única não é automaticamente uma atribuição.
- **Backtracking** experimenta um valor e desfaz a decisão quando o ramo falha.
- **MRV — Minimum Remaining Values (menor número de valores restantes)** escolhe a variável não atribuída com menos opções. Os empates seguem a ordem das variáveis indicada no exemplo.
- **LCV — Least Constraining Value (valor menos restritivo)** ordena os valores de uma variável já escolhida, contando eliminações diretas nos domínios dos vizinhos não atribuídos.
- **Forward checking** filtra os vizinhos não atribuídos depois de uma atribuição; não propaga automaticamente todas as reduções entre variáveis futuras.
- **AC-3 — Arc Consistency Algorithm 3 (algoritmo de consistência de arco n.º 3)** elimina valores sem suporte. Nesta experiência funciona antes da procura, sem atribuir valores.

Uma **tentativa** é um valor experimentado para uma variável, mesmo se rejeitado. Uma **verificação** é um teste de compatibilidade entre dois valores. As eliminações e restaurações não são tentativas. Não comparamos tempos de execução.

Cada execução começa com uma cópia dos dados iniciais. Repetir uma experiência não aproveita decisões da anterior. Voltar a executar a preparação limpa o histórico.

In [1]:
from collections import deque
from copy import deepcopy
from html import escape
import pandas as pd
from IPython.display import display, HTML

historico = []

def tabela(linhas):
    display(pd.DataFrame(linhas))

def texto_dominios(d):
    return "; ".join(f"{x}={{{', '.join(map(str, v))}}}" for x, v in d.items())

def rainhas(n=4):
    variaveis = [f"Q{i}" for i in range(1, n+1)]
    dominios = {x: list(range(1, n+1)) for x in variaveis}
    vizinhos = {x: [y for y in variaveis if y != x] for x in variaveis}
    def comp(x, a, y, b):
        return a != b and abs(a-b) != abs(int(x[1:])-int(y[1:]))
    return dominios, vizinhos, comp

def tecnicos():
    d = {"P": [1,2], "Q": [1], "R": [2,3], "S": [3,4]}
    v = {x: [y for y in d if y != x] for x in d}
    return d, v, lambda x,a,y,b: a != b

def procurar(dominios, vizinhos, comp, escolha="ordem_fixa",
             ordem_valores="crescente", forward=False, iniciais=None):
    assert escolha in ("ordem_fixa", "MRV")
    assert ordem_valores in ("crescente", "decrescente")
    d0 = deepcopy(dominios)
    atribuicao = dict(iniciais or {})
    assert all(x in d0 and a in d0[x] for x,a in atribuicao.items())
    assert all(comp(x,a,y,b) for x,a in atribuicao.items()
               for y,b in atribuicao.items() if y in vizinhos[x])
    # As decisões iniciais são dadas, não contam como tentativas.
    for x,a in atribuicao.items():
        d0[x] = [a]
    if forward:
        for y in d0:
            if y not in atribuicao:
                d0[y] = [b for b in d0[y] if all(
                    comp(y,b,x,a) for x,a in atribuicao.items() if x in vizinhos[y])]
    estat = {"Tentativas": 0, "Verificações": 0, "Valores desfeitos": 0}
    eventos = []
    def registar(acao, d, motivo=""):
        eventos.append({"Passo": len(eventos), "Ação": acao,
                        "Atribuição": str(atribuicao.copy()),
                        "Domínios por atribuir": texto_dominios({x:v for x,v in d.items() if x not in atribuicao}),
                        "Motivo": motivo})
    registar("Inicial", d0)
    def bt(d):
        if any(not vals for vals in d.values()):
            return None
        livres = [x for x in d if x not in atribuicao]
        if not livres:
            return atribuicao.copy()
        x = min(livres, key=lambda z: len(d[z])) if escolha == "MRV" else livres[0]
        for a in sorted(d[x], reverse=ordem_valores == "decrescente"):
            estat["Tentativas"] += 1
            conflito = None
            for y,b in atribuicao.items():
                if y in vizinhos[x]:
                    estat["Verificações"] += 1
                    if not comp(x,a,y,b):
                        conflito = f"Conflito com {y}={b}"
                        break
            if conflito:
                registar(f"Rejeitar {x}={a}", d, conflito)
                continue
            atribuicao[x] = a
            novo = deepcopy(d); novo[x] = [a]
            eliminados = []
            if forward:
                for y in vizinhos[x]:
                    if y in atribuicao:
                        continue
                    permitidos = []
                    for b in novo[y]:
                        estat["Verificações"] += 1
                        if comp(x,a,y,b):
                            permitidos.append(b)
                        else:
                            eliminados.append(f"{y}={b}")
                    novo[y] = permitidos
            vazios = [y for y in novo if not novo[y]]
            registar(f"Atribuir {x}={a}", novo,
                     "Eliminados: " + (", ".join(eliminados) or "nenhum"))
            if vazios:
                registar("Falha: domínio vazio", novo, ", ".join(vazios))
                resultado = None
            else:
                resultado = bt(novo)
            if resultado is not None:
                return resultado
            del atribuicao[x]
            estat["Valores desfeitos"] += 1
            registar(f"Desfazer {x}={a} e restaurar", d,
                     "Retomar o estado anterior a esta tentativa")
        registar(f"Esgotados os valores de {x}", d)
        return None
    sol = bt(d0)
    return dict(solucao=sol, eventos=eventos, **estat)

def mostrar_procura(experiencia, parametros, resultado, detalhes=True):
    resumo = {"Experiência": experiencia, "Parâmetros": parametros,
              "Solução": str(resultado["solucao"]),
              **{k:resultado[k] for k in ("Tentativas","Verificações","Valores desfeitos")}}
    historico.append(resumo)
    tabela([resumo])
    if detalhes:
        tabela(resultado["eventos"])

def tabuleiro(atribuicao, n=4):
    # Marca todas as casas atacadas, incluindo na coluna de uma rainha colocada.
    partes = ['<table style="border-collapse:collapse;text-align:center"><tr><th></th>']
    partes += [f'<th style="padding:8px">Q{i}</th>' for i in range(1,n+1)]
    partes.append('</tr>')
    for r in range(1,n+1):
        partes.append(f'<tr><th style="padding:8px">{r}</th>')
        for c in range(1,n+1):
            rainha = atribuicao.get(f"Q{c}") == r
            atacada = any(c == int(x[1:]) or r == a or abs(c-int(x[1:])) == abs(r-a)
                          for x,a in atribuicao.items())
            simbolo = '♛' if rainha else '×' if atacada else '○'
            cor = '#123652' if rainha else '#e96312' if atacada else '#19823b'
            fundo = '#e8f2f8' if (r+c)%2 == 0 else 'white'
            partes.append(f'<td style="border:1px solid #123652;width:48px;height:48px;font-size:32px;color:{cor};background:{fundo}">{simbolo}</td>')
        partes.append('</tr>')
    display(HTML(''.join(partes)+'</table>'))

def comparar_lcv(d, vizinhos, comp, iniciais, x):
    base = deepcopy(d)
    for y in base:
        if y in iniciais:
            base[y] = [iniciais[y]]
        else:
            base[y] = [b for b in base[y] if all(comp(y,b,z,a)
                       for z,a in iniciais.items() if z in vizinhos[y])]
    resultados=[]
    for a in base[x]:
        depois=deepcopy(base); depois[x]=[a]; eliminacoes=0
        for y in vizinhos[x]:
            if y in iniciais: continue
            depois[y]=[b for b in base[y] if comp(x,a,y,b)]
            eliminacoes += len(base[y])-len(depois[y])
        resultados.append({"Valor":a,"Eliminações":eliminacoes,"Depois":depois})
    return base, resultados

def ac3(dominios, ordem="normal"):
    assert ordem in ("normal", "inversa")
    d=deepcopy(dominios)
    viz={"A":["B"],"B":["A","C"],"C":["B"]}
    def comp(x,a,y,b):
        if (x,y)==("A","B"): return a>b
        if (x,y)==("B","A"): return b>a
        return a==b
    arcos=[("A","B"),("B","A"),("B","C"),("C","B")]
    fila=deque(arcos if ordem=="normal" else arcos[::-1])
    eventos=[]; verificacoes=0
    while fila:
        x,y=fila.popleft(); removidos=[]
        for a in d[x][:]:
            suporte=False
            for b in d[y]:
                verificacoes+=1
                if comp(x,a,y,b): suporte=True; break
            if not suporte:
                d[x].remove(a); removidos.append(a)
        acrescentados=[]
        falha=not d[x]
        if removidos and not falha:
            # Convenção das infografias: inclui também o arco inverso.
            for z in viz[x]:
                arco=(z,x)
                if arco not in fila:
                    fila.append(arco); acrescentados.append(f"{z}→{x}")
        eventos.append({"Arco retirado":f"{x}→{y}","Eliminados":str(removidos),
                        "Domínios":texto_dominios(d),"Acrescentados":str(acrescentados),
                        "Fila":str([f"{u}→{v}" for u,v in fila]),"Falha":falha})
        if falha:
            return d, eventos, False, verificacoes
    return d, eventos, True, verificacoes

print("Preparação concluída. O histórico foi limpo.")

Preparação concluída. O histórico foi limpo.


## Experiência 1 — Backtracking nas quatro rainhas
Uma variável por coluna; o seu valor é a linha da rainha. As rainhas não podem partilhar linha nem diagonal.

Execute com `ORDEM_VALORES="crescente"` e depois `"decrescente"`. Mantém-se a ordem Q1, Q2, Q3, Q4, sem filtragem dos domínios futuros. Os valores incompatíveis são rejeitados quando experimentados.

**Questão 1.** Qual foi a primeira solução em cada ordem? Compare tentativas e valores desfeitos. Identifique, no registo, uma variável que esgotou as alternativas e a decisão anterior que foi desfeita.

In [2]:
ORDEM_VALORES = "crescente"  # ALTERAR: "crescente", "decrescente"
d, v, comp = rainhas()
r1 = procurar(d, v, comp, ordem_valores=ORDEM_VALORES)
mostrar_procura("1", ORDEM_VALORES, r1)
if r1["solucao"]: tabuleiro(r1["solucao"])

,Experiência,Parâmetros,Solução,Tentativas,Verificações,Valores desfeitos
0,1,crescente,"{'Q1': 2, 'Q2': 4, 'Q3': 1, 'Q4': 3}",26,36,4


,Passo,Ação,Atribuição,Domínios por atribuir,Motivo
0,0,Inicial,{},"Q1={1, 2, 3, 4}; Q2={1, 2, 3, 4}; Q3={1, 2, 3,...",
1,1,Atribuir Q1=1,{'Q1': 1},"Q2={1, 2, 3, 4}; Q3={1, 2, 3, 4}; Q4={1, 2, 3, 4}",Eliminados: nenhum
2,2,Rejeitar Q2=1,{'Q1': 1},"Q2={1, 2, 3, 4}; Q3={1, 2, 3, 4}; Q4={1, 2, 3, 4}",Conflito com Q1=1
3,3,Rejeitar Q2=2,{'Q1': 1},"Q2={1, 2, 3, 4}; Q3={1, 2, 3, 4}; Q4={1, 2, 3, 4}",Conflito com Q1=1
4,4,Atribuir Q2=3,"{'Q1': 1, 'Q2': 3}","Q3={1, 2, 3, 4}; Q4={1, 2, 3, 4}",Eliminados: nenhum
5,5,Rejeitar Q3=1,"{'Q1': 1, 'Q2': 3}","Q3={1, 2, 3, 4}; Q4={1, 2, 3, 4}",Conflito com Q1=1
6,6,Rejeitar Q3=2,"{'Q1': 1, 'Q2': 3}","Q3={1, 2, 3, 4}; Q4={1, 2, 3, 4}",Conflito com Q2=3
7,7,Rejeitar Q3=3,"{'Q1': 1, 'Q2': 3}","Q3={1, 2, 3, 4}; Q4={1, 2, 3, 4}",Conflito com Q1=1
8,8,Rejeitar Q3=4,"{'Q1': 1, 'Q2': 3}","Q3={1, 2, 3, 4}; Q4={1, 2, 3, 4}",Conflito com Q2=3
9,9,Esgotados os valores de Q3,"{'Q1': 1, 'Q2': 3}","Q3={1, 2, 3, 4}; Q4={1, 2, 3, 4}",


,Q1,Q2,Q3,Q4
1,×,×,♛,×
2,♛,×,×,×
3,×,×,×,♛
4,×,♛,×,×


**Resposta 1:**

| Ordem | Solução (Q1,Q2,Q3,Q4) | Tentativas | Valores desfeitos |
|---|---|---|---|
| Crescente | … | … | … |
| Decrescente | … | … | … |

Exemplo de esgotamento e recuo: …

Interpretação da comparação: …

## Experiência 2 — Detetar a falha antes de colocar a próxima rainha
Retomamos o ramo com **Q1=1 já dado**. Essa atribuição inicial não conta como tentativa. Mantemos ordem fixa e valores crescentes.

Execute com `FORWARD_CHECKING=False` e depois `True`. Observe em particular a tentativa **Q2=3**. O registo mostra domínios por atribuir: sem forward checking, estes não são filtrados, mesmo que alguns valores já sejam incompatíveis com decisões anteriores.

**Questão 2.** Com Q1=1 e Q2=3, como é detetada a falha em cada versão? Qual o domínio que fica vazio com forward checking? Identifique o estado restaurado antes de tentar Q2=4. Compare o trabalho e diga se existe solução com Q1=1 fixado.

In [3]:
FORWARD_CHECKING = False  # ALTERAR: False, True
d, v, comp = rainhas()
r2 = procurar(d, v, comp, forward=FORWARD_CHECKING, iniciais={"Q1":1})
mostrar_procura("2", f"FC={FORWARD_CHECKING}; Q1=1 fixo", r2)
print("Tentativa em análise: Q1=1, Q2=3")
tabuleiro({"Q1":1,"Q2":3})

,Experiência,Parâmetros,Solução,Tentativas,Verificações,Valores desfeitos
0,2,FC=False; Q1=1 fixo,None,16,23,3


,Passo,Ação,Atribuição,Domínios por atribuir,Motivo
0,0,Inicial,{'Q1': 1},"Q2={1, 2, 3, 4}; Q3={1, 2, 3, 4}; Q4={1, 2, 3, 4}",
1,1,Rejeitar Q2=1,{'Q1': 1},"Q2={1, 2, 3, 4}; Q3={1, 2, 3, 4}; Q4={1, 2, 3, 4}",Conflito com Q1=1
2,2,Rejeitar Q2=2,{'Q1': 1},"Q2={1, 2, 3, 4}; Q3={1, 2, 3, 4}; Q4={1, 2, 3, 4}",Conflito com Q1=1
3,3,Atribuir Q2=3,"{'Q1': 1, 'Q2': 3}","Q3={1, 2, 3, 4}; Q4={1, 2, 3, 4}",Eliminados: nenhum
4,4,Rejeitar Q3=1,"{'Q1': 1, 'Q2': 3}","Q3={1, 2, 3, 4}; Q4={1, 2, 3, 4}",Conflito com Q1=1
5,5,Rejeitar Q3=2,"{'Q1': 1, 'Q2': 3}","Q3={1, 2, 3, 4}; Q4={1, 2, 3, 4}",Conflito com Q2=3
6,6,Rejeitar Q3=3,"{'Q1': 1, 'Q2': 3}","Q3={1, 2, 3, 4}; Q4={1, 2, 3, 4}",Conflito com Q1=1
7,7,Rejeitar Q3=4,"{'Q1': 1, 'Q2': 3}","Q3={1, 2, 3, 4}; Q4={1, 2, 3, 4}",Conflito com Q2=3
8,8,Esgotados os valores de Q3,"{'Q1': 1, 'Q2': 3}","Q3={1, 2, 3, 4}; Q4={1, 2, 3, 4}",
9,9,Desfazer Q2=3 e restaurar,{'Q1': 1},"Q2={1, 2, 3, 4}; Q3={1, 2, 3, 4}; Q4={1, 2, 3, 4}",Retomar o estado anterior a esta tentativa


Tentativa em análise: Q1=1, Q2=3


,Q1,Q2,Q3,Q4
1,♛,×,×,×
2,×,×,×,○
3,×,♛,×,×
4,×,×,×,×


**Resposta 2:**

| Forward checking | Tentativas | Verificações | Existe solução neste ramo? |
|---|---|---|---|
| False | … | … | … |
| True | … | … | … |

Deteção da falha com Q2=3, nas duas versões: …

Domínios restaurados de Q2, Q3 e Q4; atribuição mantida: …

Porque esta falha não prova que o problema das quatro rainhas seja impossível: …

## Experiência 3 — MRV escolhe a tarefa
Quatro tarefas decorrem no mesmo período. Cada técnico só pode realizar uma tarefa. Os domínios já refletem as competências:

| Tarefa | Técnicos habilitados |
|---|---|
| P | {1,2} |
| Q | {1} |
| R | {2,3} |
| S | {3,4} |

Aplicamos as desigualdades entre todos os pares de tarefas. Mantenha forward checking e valores crescentes. Execute com `"ordem_fixa"` (P,Q,R,S) e depois `"MRV"`. MRV desempata pela mesma ordem.

**Questão 3.** Qual é a primeira tarefa escolhida em cada execução? Compare tentativas e explique a falha evitada por MRV. O resultado permite concluir que MRV reduz sempre o trabalho?

In [4]:
ESCOLHA_VARIAVEL = "ordem_fixa"  # ALTERAR: "ordem_fixa", "MRV"
d, v, comp = tecnicos()
r3 = procurar(d, v, comp, escolha=ESCOLHA_VARIAVEL, forward=True)
mostrar_procura("3", ESCOLHA_VARIAVEL, r3)

,Experiência,Parâmetros,Solução,Tentativas,Verificações,Valores desfeitos
0,3,ordem_fixa,"{'P': 2, 'Q': 1, 'R': 3, 'S': 4}",5,21,1


,Passo,Ação,Atribuição,Domínios por atribuir,Motivo
0,0,Inicial,{},"P={1, 2}; Q={1}; R={2, 3}; S={3, 4}",
1,1,Atribuir P=1,{'P': 1},"Q={}; R={2, 3}; S={3, 4}",Eliminados: Q=1
2,2,Falha: domínio vazio,{'P': 1},"Q={}; R={2, 3}; S={3, 4}",Q
3,3,Desfazer P=1 e restaurar,{},"P={1, 2}; Q={1}; R={2, 3}; S={3, 4}",Retomar o estado anterior a esta tentativa
4,4,Atribuir P=2,{'P': 2},"Q={1}; R={3}; S={3, 4}",Eliminados: R=2
5,5,Atribuir Q=1,"{'P': 2, 'Q': 1}","R={3}; S={3, 4}",Eliminados: nenhum
6,6,Atribuir R=3,"{'P': 2, 'Q': 1, 'R': 3}",S={4},Eliminados: S=3
7,7,Atribuir S=4,"{'P': 2, 'Q': 1, 'R': 3, 'S': 4}",,Eliminados: nenhum


**Resposta 3:**

| Estratégia | Primeira tarefa | Solução (P,Q,R,S) | Tentativas | Verificações |
|---|---|---|---|---|
| Ordem fixa | … | … | … | … |
| MRV | … | … | … | … |

Falha evitada e motivo: …

Limites da conclusão sobre MRV: …

## Experiência 4 — LCV ordena as linhas de Q2
Mantemos **Q1=1** e fixamos **Q2 como próxima variável**. Após forward checking, D(Q2)={3,4}, D(Q3)={2,4} e D(Q4)={2,3}.

Execute com `VALOR_Q2=3` e depois `4`. A comparação LCV conta apenas opções eliminadas de **Q3 e Q4**, partindo sempre do mesmo estado. Não conta casas da coluna de uma rainha já colocada nem opções que já tinham sido eliminadas.

**Questão 4.** Quantas opções elimina cada alternativa? Qual é preferida por LCV? No caso Q2=4, as opções restantes Q3=2 e Q4=3 são compatíveis entre si? Explique por que a escolha LCV não garante solução.

In [5]:
VALOR_Q2 = 3  # ALTERAR: 3, 4
assert VALOR_Q2 in (3,4)
d, v, comp = rainhas()
base, alternativas = comparar_lcv(d, v, comp, {"Q1":1}, "Q2")
tabela([{"Q2":a["Valor"],"Opções eliminadas":a["Eliminações"],
         "D(Q3)":str(a["Depois"]["Q3"]),"D(Q4)":str(a["Depois"]["Q4"])} for a in alternativas])
a = next(a for a in alternativas if a["Valor"] == VALOR_Q2)
print("Domínios antes:", texto_dominios(base))
print("Domínios depois:", texto_dominios(a["Depois"]))
tabuleiro({"Q1":1,"Q2":VALOR_Q2})
historico.append({"Experiência":"4","Parâmetros":f"Q2={VALOR_Q2}",
                  "Opções eliminadas":a["Eliminações"],"Domínios":texto_dominios(a["Depois"])})

,Q2,Opções eliminadas,D(Q3),D(Q4)
0,3,3,[],[2]
1,4,2,[2],[3]


Domínios antes: Q1={1}; Q2={3, 4}; Q3={2, 4}; Q4={2, 3}
Domínios depois: Q1={1}; Q2={3}; Q3={}; Q4={2}


,Q1,Q2,Q3,Q4
1,♛,×,×,×
2,×,×,×,○
3,×,♛,×,×
4,×,×,×,×


**Resposta 4:**

| Valor de Q2 | D(Q3) depois | D(Q4) depois | Total de opções eliminadas |
|---|---|---|---|
| 3 | … | … | … |
| 4 | … | … | … |

Valor preferido e justificação: …

Compatibilidade de Q3=2 com Q4=3: …

Diferença entre o papel de MRV e o de LCV: …

## Experiência 5 — AC-3 reduz domínios sem atribuir valores
Usamos A>B e B=C, com D(A)=D(B)={1,2,3}. O domínio inicial de C pode ser alterado.

Execute estas **três configurações**:
1. `DOMINIO_C=[1,2,3]`, `ORDEM_FILA="normal"`;
2. `DOMINIO_C=[1,2,3]`, `ORDEM_FILA="inversa"`;
3. `DOMINIO_C=[3]`, `ORDEM_FILA="normal"`.

A fila normal é [A→B, B→A, B→C, C→B]. Rever X→Y só pode eliminar valores de X, procurando suporte em Y.

Como nas infografias, quando X muda voltamos a acrescentar **todos os arcos que consultam X, incluindo o inverso**, sem duplicar os pendentes. Esta variante pode fazer revisões redundantes; a versão habitual pode excluir o inverso. Nenhuma atribuição é feita e não procuramos soluções completas.

**Questão 5.** Registe os domínios no fim e o número de arcos revistos. No cenário original, a ordem da fila muda os domínios finais? No cenário C={3}, identifique a eliminação que provoca um domínio vazio. Distinga fila vazia de domínio vazio.

In [6]:
DOMINIO_C = [1,2,3]  # ALTERAR: [1,2,3] ou [3]
ORDEM_FILA = "normal"  # ALTERAR: "normal", "inversa"
assert DOMINIO_C in ([1,2,3],[3])
d5, passos5, sucesso5, verificacoes5 = ac3({"A":[1,2,3],"B":[1,2,3],"C":DOMINIO_C}, ORDEM_FILA)
tabela(passos5)
print("Domínios no fim:", texto_dominios(d5))
print("Fila esgotada sem domínio vazio" if sucesso5 else "FALHA: foi detetado um domínio vazio")
print("Arcos revistos:", len(passos5), "| Verificações de pares:", verificacoes5)
historico.append({"Experiência":"5","Parâmetros":f"C={DOMINIO_C}; fila={ORDEM_FILA}",
                  "Domínios":texto_dominios(d5),"Arcos revistos":len(passos5),
                  "Verificações":verificacoes5,"Falha":not sucesso5})

,Arco retirado,Eliminados,Domínios,Acrescentados,Fila,Falha
0,A→B,[1],"A={2, 3}; B={1, 2, 3}; C={1, 2, 3}",[],"['B→A', 'B→C', 'C→B']",False
1,B→A,[3],"A={2, 3}; B={1, 2}; C={1, 2, 3}",['A→B'],"['B→C', 'C→B', 'A→B']",False
2,B→C,[],"A={2, 3}; B={1, 2}; C={1, 2, 3}",[],"['C→B', 'A→B']",False
3,C→B,[3],"A={2, 3}; B={1, 2}; C={1, 2}",['B→C'],"['A→B', 'B→C']",False
4,A→B,[],"A={2, 3}; B={1, 2}; C={1, 2}",[],['B→C'],False
5,B→C,[],"A={2, 3}; B={1, 2}; C={1, 2}",[],[],False


Domínios no fim: A={2, 3}; B={1, 2}; C={1, 2}
Fila esgotada sem domínio vazio
Arcos revistos: 6 | Verificações de pares: 23


**Resposta 5:**

| D(C) inicial | Ordem da fila | D(A) no fim | D(B) no fim | D(C) no fim | Arcos revistos | Falha? |
|---|---|---|---|---|---|---|
| {1,2,3} | Normal | … | … | … | … | … |
| {1,2,3} | Inversa | … | … | … | … | … |
| {3} | Normal | … | … | … | … | … |

Efeito da ordem das verificações no cenário original: …

Arco e eliminação que detetam a falha no cenário C={3}: …

Diferença entre fila vazia e domínio vazio: …

Porque os domínios finais não são uma atribuição completa: …

In [ ]:
display(pd.DataFrame(historico).fillna("—"))

,Experiência,Parâmetros,Solução,Tentativas,Verificações,Valores desfeitos,Opções eliminadas,Domínios,Arcos revistos,Falha
0,1,crescente,"{'Q1': 2, 'Q2': 4, 'Q3': 1, 'Q4': 3}",26.0,36.0,4.0,—,—,—,—
1,2,FC=False; Q1=1 fixo,None,16.0,23.0,3.0,—,—,—,—
2,3,ordem_fixa,"{'P': 2, 'Q': 1, 'R': 3, 'S': 4}",5.0,21.0,1.0,—,—,—,—
3,4,Q2=3,—,—,—,—,3.0,Q1={1}; Q2={3}; Q3={}; Q4={2},—,—
4,5,"C=[1, 2, 3]; fila=normal",—,—,23.0,—,—,"A={2, 3}; B={1, 2}; C={1, 2}",6.0,False
